# 05 Hybrid Skill Coverage

Purpose: complement the embedding baseline with explicit job-skill coverage and build the hybrid ranking matrix.

Inputs:
- `data/processed/degree_profiles.parquet`
- `data/interim/jobs_clean.parquet`
- `data/embeddings/similarity_matrix_all-MiniLM-L6-v2.npy`

Outputs:
- `data/processed/degree_skills.json`
- `data/embeddings/skill_coverage_matrix.npy`
- `data/embeddings/hybrid_matrix.npy`


In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.io import paths, require_files

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

from analysis.embedding import top_k_jobs_for_degree
from analysis.hybrid import build_hybrid_matrix, build_skill_vocab, compute_skill_coverage_matrix, load_or_build_degree_skills

ALPHA = 0.7
BETA = 0.3
MODEL_NAME = 'all-MiniLM-L6-v2'

require_files([paths.degree_profiles_parquet, paths.jobs_clean_parquet, paths.similarity_matrix_npy(MODEL_NAME)])
print('Expected upstream inputs: baseline similarity matrix plus cleaned jobs and degree profiles')


Expected upstream inputs: baseline similarity matrix plus cleaned jobs and degree profiles


In [2]:
degree_profiles = pd.read_parquet(paths.degree_profiles_parquet)
jobs = pd.read_parquet(paths.jobs_clean_parquet)
sim_matrix = np.load(paths.similarity_matrix_npy(MODEL_NAME))

skill_vocab = build_skill_vocab(jobs)
degree_skills = load_or_build_degree_skills(degree_profiles, skill_vocab, paths.degree_skills_json)

if paths.skill_coverage_matrix_npy.exists():
    skill_coverage_matrix = np.load(paths.skill_coverage_matrix_npy)
    if skill_coverage_matrix.shape != sim_matrix.shape:
        skill_coverage_matrix = compute_skill_coverage_matrix(degree_profiles, jobs, degree_skills)
        np.save(paths.skill_coverage_matrix_npy, skill_coverage_matrix)
else:
    skill_coverage_matrix = compute_skill_coverage_matrix(degree_profiles, jobs, degree_skills)
    np.save(paths.skill_coverage_matrix_npy, skill_coverage_matrix)

if paths.hybrid_matrix_npy.exists():
    hybrid_matrix = np.load(paths.hybrid_matrix_npy)
    if hybrid_matrix.shape != sim_matrix.shape:
        hybrid_matrix = build_hybrid_matrix(sim_matrix, skill_coverage_matrix, alpha=ALPHA, beta=BETA)
        np.save(paths.hybrid_matrix_npy, hybrid_matrix)
else:
    hybrid_matrix = build_hybrid_matrix(sim_matrix, skill_coverage_matrix, alpha=ALPHA, beta=BETA)
    np.save(paths.hybrid_matrix_npy, hybrid_matrix)

print(f'Unique skills in jobs: {len(skill_vocab):,}')
print(f'Skill coverage matrix shape: {skill_coverage_matrix.shape}')
print(f'Hybrid matrix shape: {hybrid_matrix.shape}')


Unique skills in jobs: 0
Skill coverage matrix shape: (5, 17346)
Hybrid matrix shape: (5, 17346)


In [3]:
def top_k_jobs_hybrid(degree_id: str, k: int = 10) -> pd.DataFrame:
    matches = degree_profiles.index[degree_profiles['degree_id'].astype(str) == str(degree_id)]
    if len(matches) == 0:
        raise KeyError(f'Degree id {degree_id!r} not found.')
    degree_idx = degree_profiles.index.get_loc(matches[0])
    scores = hybrid_matrix[degree_idx]
    top_idx = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({
        'rank': range(1, len(top_idx) + 1),
        'job_id': jobs.iloc[top_idx]['job_id'].values,
        'job_title': jobs.iloc[top_idx]['title'].values,
        'company': jobs.iloc[top_idx]['company'].values,
        'categories': jobs.iloc[top_idx]['categories_str'].values,
        'cosine_sim': sim_matrix[degree_idx, top_idx].round(4),
        'skill_coverage': skill_coverage_matrix[degree_idx, top_idx].round(4),
        'hybrid_score': scores[top_idx].round(4),
    })

for degree_id in ['cs', 'dsa']:
    if (degree_profiles['degree_id'] == degree_id).any():
        print(f'Baseline top jobs for {degree_id}:')
        display(top_k_jobs_for_degree(sim_matrix, degree_profiles, jobs, degree_id, 'sim_score', k=5))
        print(f'Hybrid top jobs for {degree_id}:')
        display(top_k_jobs_hybrid(degree_id, k=5))


Baseline top jobs for cs:


,rank,job_id,job_title,company,categories,sim_score
0,1,MCF-2025-1715506,Software Development Engineer,NUTEK PRIVATE LIMITED,"Engineering, Manufacturing",0.4751
1,2,MCF-2025-1801982,Software Engineer (Precision / Automation),WECRUIT PTE. LTD.,"Engineering, Information Technology, Manufactu...",0.4723
2,3,MCF-2025-1801975,Software Engineer (Semiconductor Industry),WECRUIT PTE. LTD.,"Engineering, Information Technology, Manufactu...",0.4700
3,4,MCF-2026-0190683,Software Application Engineer - WCAN,WECRUIT PTE. LTD.,Information Technology,0.4629
4,5,MCF-2026-0188112,"Software Senior /Engineer (C#,PLC)- Automated ...",PEOPLE PROFILERS PTE. LTD.,"Engineering, Sciences / Laboratory / R&D",0.4602


Hybrid top jobs for cs:


,rank,job_id,job_title,company,categories,cosine_sim,skill_coverage,hybrid_score
0,1,MCF-2025-1715506,Software Development Engineer,NUTEK PRIVATE LIMITED,"Engineering, Manufacturing",0.4751,0.0,0.3326
1,2,MCF-2025-1801982,Software Engineer (Precision / Automation),WECRUIT PTE. LTD.,"Engineering, Information Technology, Manufactu...",0.4723,0.0,0.3306
2,3,MCF-2025-1801975,Software Engineer (Semiconductor Industry),WECRUIT PTE. LTD.,"Engineering, Information Technology, Manufactu...",0.4700,0.0,0.3290
3,4,MCF-2026-0190683,Software Application Engineer - WCAN,WECRUIT PTE. LTD.,Information Technology,0.4629,0.0,0.3240
4,5,MCF-2026-0188112,"Software Senior /Engineer (C#,PLC)- Automated ...",PEOPLE PROFILERS PTE. LTD.,"Engineering, Sciences / Laboratory / R&D",0.4602,0.0,0.3221


Baseline top jobs for dsa:


,rank,job_id,job_title,company,categories,sim_score
0,1,MCF-2025-1604902,Data Analyst,MSI GLOBAL PRIVATE LIMITED,"Engineering, Information Technology, Others",0.5470
1,2,MCF-2026-0186942,Principal Data Scientist (Aerospace),RN CARE PTE. LTD.,"Engineering, Information Technology, Manufactu...",0.5441
2,3,MCF-2025-1424769,Data Engineer,DEEGIT ASIA PTE. LTD.,Information Technology,0.5233
3,4,MCF-2025-1164543,Outreach Data Analyst,PERSOL SINGAPORE PTE. LTD.,Public / Civil Service,0.5174
4,5,MCF-2026-0161241,Senior Data Analyst (Bank | Up to 9.5k),ADECCO PERSONNEL PTE LTD,Banking and Finance,0.5087


Hybrid top jobs for dsa:


,rank,job_id,job_title,company,categories,cosine_sim,skill_coverage,hybrid_score
0,1,MCF-2025-1604902,Data Analyst,MSI GLOBAL PRIVATE LIMITED,"Engineering, Information Technology, Others",0.5470,0.0,0.3829
1,2,MCF-2026-0186942,Principal Data Scientist (Aerospace),RN CARE PTE. LTD.,"Engineering, Information Technology, Manufactu...",0.5441,0.0,0.3808
2,3,MCF-2025-1424769,Data Engineer,DEEGIT ASIA PTE. LTD.,Information Technology,0.5233,0.0,0.3663
3,4,MCF-2025-1164543,Outreach Data Analyst,PERSOL SINGAPORE PTE. LTD.,Public / Civil Service,0.5174,0.0,0.3622
4,5,MCF-2026-0161241,Senior Data Analyst (Bank | Up to 9.5k),ADECCO PERSONNEL PTE LTD,Banking and Finance,0.5087,0.0,0.3561
